## Sintezavimo sprendimo instrukcija

Žemiau pateiktos techninės instrukcijos ir skriptai, skirtos lietuviškos TTS sistemos administratoriui, kurie parodo, kaip galima kurti lokaliai (vietiniame kompiuteryje) veikiantį TTS sprendimą, skirtą sintezuoti lietuvišką šneką. Vietiniame kompiuteryje reikia turėti:
1. Akustinio modelio archyvą
2. Vokoderio archyvą
   
### 0. Inicializuokite pagalbines funkcijas

In [ ]:
# Inicializuokite pagalbines funkcijas

from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from espnet2.main_funcs.pack_funcs import unpack, find_path_and_change_it_recursive
import tarfile
from pathlib import Path
from typing import Dict
import yaml
import time

cache_dir = "./.cache/"
device = "cpu"
#device = "cuda" # uncomment if you have a GPU and want to use it

def _safe_extract_tar(tar: tarfile.TarFile, path: Path):
    base = path.resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise Exception("Unsafe tar archive")

    tar.extractall(path)

def find_vocoder(search_dir: Path) -> str:
    for file in search_dir.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError(f"No *.pkl file found in the {search_dir}")    


def get_dict_from_cache(meta: Path) -> Dict[str, str]:
    if not meta.exists():
        raise FileNotFoundError(f"No meta.yaml found in the {meta}")
        
    outpath = meta.parent
    with meta.open("r", encoding="utf-8") as f:
        d = yaml.safe_load(f)
        yaml_files = d["yaml_files"]
        files = d["files"]

        for _, value in yaml_files.items():
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            with (outpath / value).open("r", encoding="utf-8") as f:
                d_int = yaml.safe_load(f)
            for path in outpath.rglob("*"):
                if path.is_file():
                    relative_path = path.relative_to(outpath)
                    d_int = find_path_and_change_it_recursive(d_int, str(relative_path), str(path))    
            with (outpath / value).open("w", encoding="utf-8") as f:
                yaml.safe_dump(d_int, f)
        retval = {}
        for key, value in list(yaml_files.items()) + list(files.items()):
            if not (outpath / value).exists():
                raise FileNotFoundError(f"No {outpath / value} found")
            retval[key] = str(outpath / value)
        return retval
        

def extract_archive_cached(archive_path: str, cache_dir: str) -> Path:
    archive_path = Path(archive_path)
    if archive_path.is_dir():
        return archive_path  # already extracted
    
    cache_dir = Path(cache_dir)

    stat = archive_path.stat()

    # fast cache key (no hashing)
    key = f"{archive_path.name}_{stat.st_size}_{int(stat.st_mtime)}"

    extract_path = cache_dir / key
    marker = extract_path / ".done"

    if marker.exists():
        return extract_path

    extract_path.mkdir(parents=True, exist_ok=True)

    if archive_path.name.endswith(".tar.gz"):
        with tarfile.open(archive_path) as tar:
            _safe_extract_tar(tar, extract_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    marker.touch()
    return extract_path
    

def init_tts(am_file=None, vocoder_file=None):
    if not vocoder_file:
        vocoder_file = None # make sure it's None, not empty string
    else:
        vocoder_dir = extract_archive_cached(archive_path=vocoder_file, cache_dir=cache_dir)
        vocoder_file = find_vocoder(vocoder_dir)
        print(f"vocoder_file: {vocoder_file}")

    print(f"\n===================================\nAM      = {am_file}")
    print(f"Vocoder = {vocoder_file if vocoder_file else 'Griffin-Lim'}")
    print(f"===================================")

    am_path = Path(am_file)
    if am_path.is_dir():
        meta = am_path / "meta.yaml"
        kwargs = get_dict_from_cache(meta=meta)
    else:    
        kwargs = unpack(input_archive = am_file, outpath = cache_dir)
    print (kwargs)
    
    tts = Text2Speech.from_pretrained(
        vocoder_file=vocoder_file, 
        device=device,
        **kwargs
    )
    def run_tts(text):
        started = time.perf_counter()
        res = tts(text)
        audio = res["wav"].cpu().numpy()
        generation_seconds = time.perf_counter() - started
        audio_duration = len(audio) / tts.fs
        print(f'  Duration: {audio_duration:.2f}s')
        print(f'  In: {generation_seconds:.2f}s')
        print(f'  RTF: {generation_seconds / audio_duration:.3f}x')
        return audio
    return run_tts, tts.fs

## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser     

print ("ESPnet inicializuotas")    

### 1. Modelis

Nurodykite kelią iki Akustinio modelio ir Vokoderio

In [ ]:
# Nurodykite kelią iki akustinio modelio zip failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.:  git clone https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01
# am_zip_file_or_dir="./data/v01/ner.tts_train_fastspeech2_raw_phn_none_train.loss.ave.zip"
am_zip_file_or_dir="./data/sab/v01/tts_train_fastspeech2_raw_phn_none_train.loss.ave.zip"

# Nurodykite kelią iki vokoderio tar gz failo
# arba iki direktorijos, jei parsisiuntėte iš HuggingFace
# pvz.: git clone https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01

# arba palikite tuščią, jei norite naudoti Griffin-Lim algoritmą
vocoder_tar_gz_file_or_dir="./data/sabina.style.v04-1000000.tar.gz"
# vocoder_tar_gz_file_or_dir="sing-agn.vocoder.style_melgan.v01"
# vocoder_tar_gz_file_or_dir=None

# bus išskleisti į './.cache/' katalogą"
print (f"AM zip failas  : {am_zip_file_or_dir}")
print (f"Vocoder failas : {vocoder_tar_gz_file_or_dir}")

# Inicializuokite TTS modelį
tts, fs = init_tts(am_file=am_zip_file_or_dir, vocoder_file=vocoder_tar_gz_file_or_dir)
print (f"\n\nREADY: modelis įkeltas. FS: {fs}Hz\n") 

### 2. Sintezavimas

Pakoreguokite/įveskite sakinius, kuriuos norite perskaityti

In [ ]:
text = """sil p ^o: v a k a r' ^ie n' E: s k l' e b ^o: n a s' v' E: l. p "a l' ie p' E: k l' ^ie r' i k ui p' e t' r' "i: l ai r ^uo S' t' i v' "i s a , k "a s' r' ei k a l' "i N. g a d "a: r. b ui . sil S' "i s' j ^eu Z' i n "o: j io: s "a v o: k l' e b o: n' "i j io: s t v a ^r. k a: , i S t r "au k' E: ^i: v' "i d u r' i: s a l "o n o: p' "e: r.' l' e N.' k' e m a: , Z a l' "e g' ^e: l u m.' b' e "i S m u S t a: s t a l' ^e: l' i: , "i S' E: m' E: k r' ei d ^o: s , p a d' "E: j io: "u Z' d' e k t a Z v a k' "e s i ^r. "a t' n' e S' E: k o: r. t "a s . sil
sil t' "i g d a b a ^r. v a s ^a: r' i s u p r ^a: t o: , k "a s' tS' e b "u s p' e r. d "a: r. b a s . sil a p' l' i ^N. k s t a l' ^e: l' i: s u s' "E: d o: p a t s k l' e b ^o: n a s i ^r.' t' r' ^i: s' v' e tS' ^ei . sil
sil i S s ^o: d o: j ^ie d u m ^a: t' E: , k "a t' v' i r. t "u v' E: i t ^ai p "a t' ^ei n a d' "i d' e l' i z d "a: r. b a s , s t u k s' "E: j io: p' ^ei l' ei , b a r.' S' k' "E: j io: i ^n. d ai , k a S k ^a: tS' "i r.' S' k' i n o: , k a S k ^a: v' "i r' E: i ^r.' k' ^e: p' E: . sil v a s ^a: r' iui b "u v o: i: d o: m "u i Z' g' i ^r.' s' t' i k ^a: n "o: r. s i S s "a v o: d r ^au g o: "a p' ie v' ^ie t o: s k "u n' i g u s i ^r. k l' e b o: n' "i j e: i ^r.' j "i s , p r a d' "E: d a m a s k a ^l. b a: , t ^a: r' E: , sil
sil v a s ^a: r' i s p a j "iu t o: , k "a d r ^au g o: Z ^o: dZ' iuo s' e ^e: s a m a k a S k o: k' ^iu: i: t a r' "i m u: , b' "e t k l au s' i n' "E: t' i p a "ai S' k' i n' i m u: i ^r. s m "u l.' k m' e n u: j "e: m. p a s' i r "o: d' E: b "u: s' e n.' t' n' e p a d o: r "u , i ^r.' j ^ie , t o: l' ^eu t ^a: t' "e m a: t' e: z d a m' "i , n' e i S' v' "e: N.' k s' e: a p k a l.' b' "E: j i m o: i ^r.' v' i: r' e s' n' ^iu: j iu: k r' i t' i k ^a: v' i m o: . sil
sil b' "e t ^o: , j i s ^ai m ^a: t' E: , k "a t' i ^r.' p' e t' r' "i: l a ^i: k l "au s' i m u s n' e n "o: r o: m' i s a ts' i l' ^ie p' e . sil t' i: l' "E: d a m' i j ^ie d u ^E: j io: t o: l' ^eu t a k "u , i S a b' ie j ^iu: p "u s' iu: a p "au g u s' iu au k S t ^ai s' e r.' b' e ^n. t u: k r "u: m ai s . sil n a k t' "i z b "u v o: t' i: l' "i i ^r.' S' i l. t "a . sil
sil p' e t' r' "i: l a t' "i: tS' e g a ^r.' s' ei s u k "o: s' E: j io: i ^r. , n u t' v' "E: r' e: s v a s ^a: r' i: u s k v' e ^r. n o: , n "u t' e m.' p' E: n ^uo t ^a: k o: ^i: k r "u: m u s . sil p ^o: v a l a n.' d' ^E: l' E: s j ^ie d u p a m ^a: t' E: , k ^ai p' S' v' ie s' "i i ^r. t a m.' s' "i f' i g u: r "a n u ^E: j io: ^i: p' r' "ie S' i N. g a s p u s' "e s . sil
"""
texts = text.split("\n")
for i, text in enumerate(texts):
    if text:
        print(f"\n==========================================================================\nText = {text}")
        wav = tts(text)
        filename = f"output_senas_{i}.wav"
        sf.write(filename, wav, fs)
        display(Audio(filename))

   
